# Simulation-driven genetic algorithm

Evolves graph topologies where the fitness is **measured by simulation**, not predicted by
a regressor. Each generation is its own simulation batch under
`simulation_data/ga_runs/<run>/generations/gen_NNN/`.

This notebook only **launches** runs and **reads** their results. The loop itself lives in
`moran_process.pipeline.ga_search` and runs as an LSF job, because a run takes hours and
must survive a dropped kernel or a preemption.

The ML-predicted-fitness version is `extreme_graphs.ipynb`, kept for comparison.
See `GA_SIMULATION_PLAN.md` for why each parameter is what it is.

In [ ]:
%load_ext autoreload
%autoreload 2
%cd /home/labs/pilpel/matanyaw/moran-process

import sys

sys.path.insert(0, "src")

from pathlib import Path

import joblib
import pandas as pd

from moran_process.pipeline import ga_search
from moran_process.analysis.analysis_utils import ga_io, ga_plots

GA_RUNS_DIR = Path("simulation_data/ga_runs")
# PREFIX = "2026_07_28-long-100-gen-run"     # groups one launch of the 2x2 matrix
PREFIX = "2026_08_04-smoke-test-2"

# The batch whose random (31, 34) graphs the winners are compared against.
REFERENCE_BATCH = Path("simulation_data/2026_07_28-respiratory-vs-random-10K-reps-3")

## 1. Launch

Four runs: `{mean_steps, prob_fixation} x {maximize, minimize}`, submitted as four
concurrent driver jobs so the wall clock is one run's, not four.

Within a replicate they share one initial-population seed, so all four start from the
identical 20 random (31, 34) graphs and any divergence between them is attributable to the
objective alone. Set `replicates=3` to repeat the whole matrix independently: replicate
*k* gets `seed + k*1000`, which reseeds the starting population, the mutation stream and
the per-task simulation seeds together. Section 9 compares them.

`n_repeats = 100_000` is set by the noisier of the two metrics. What it has to buy is
**selection efficiency**, the correlation between measured and true fitness,

$$\rho = 1 / \sqrt{1 + (\mathrm{SEM}/\mathrm{SD_{between}})^2}$$

since the per-generation response to selection is proportional to it. For `prob_fixation`
at 100K repeats that is ~0.85 and it stays there for all 100 generations (section 8);
at 10K it would be ~0.80 at the start and fall as the population converges, and at 1K it
is 0.39, which still improves but roughly 2.5x slower per generation.

**Re-running this cell resubmits.** A run directory that already exists is *resumed*, not
restarted, which is what makes an LSF preemption harmless. Pass `extra_args=["--force"]`
to start over.

Set `NTFY_TOPIC` in the shell before launching (it is forwarded to the driver) and each
run pushes a notification to <https://ntfy.sh/> when it finishes or dies.

In [ ]:
# Running a toy GA

jobs = ga_search.submit_all_runs(
    GA_RUNS_DIR,
    prefix=PREFIX,
    generations=3,
    pop_size=5,
    n_children=4,        # -> 25 candidates/generation
    n_repeats=10_000,
    seed=42,
    replicates=1,
    queue="gsla-cpu",
)
jobs

In [ ]:
# jobs = ga_search.submit_all_runs(
#     GA_RUNS_DIR,
#     prefix=PREFIX,
#     generations=100,
#     pop_size=20,          # elites kept
#     n_children=10,        # mutants per elite -> 220 candidates/generation
#     n_repeats=100_000,
#     seed=42,              # shared within a replicate, offset by 1000 between replicates
#     replicates=1,         # >1 repeats the whole 2x2 matrix independently
#     queue="gsla-cpu",     # for the per-generation simulation arrays
# )
# jobs

## 2. Progress

Safe to run at any time, including mid-flight: it reads each run's `ga_state.json` and
nothing else. The driver runs detached, so this is how you see where the search is.

In [ ]:
# RUNS comes from the launch cell's return value when this kernel did the launching, and
# falls back to finding runs on disk otherwise (reading a search launched days ago, or
# after a kernel restart). Preferring `jobs` removes a whole class of confusion: editing
# PREFIX after launching used to leave the glob matching nothing, and every cell below
# would then quietly report on zero runs.
#
# load_ga_runs identifies runs by the presence of ga_config.json rather than by name, and
# raises if the prefix matches nothing, so a typo says so instead of looking like a search
# that has not started yet.
RUNS = (
    [GA_RUNS_DIR / name for name in jobs]
    if "jobs" in dir()
    else ga_io.load_ga_runs(GA_RUNS_DIR, prefix=PREFIX)
)

ga_io.ga_progress(RUNS)

In [ ]:
# Anything the driver had to warn about (a generation that came back short and was
# automatically resubmitted). Empty is the expected result.
for run in RUNS:
    state = ga_io.load_ga_state(run)
    if state and state.get("warnings"):
        print(run.name, state["warnings"])

## 3. Fitness trajectories

Same figure as `plot_multi_model_history` in `extreme_graphs.ipynb`: left axis fixation
time, right axis fixation probability, solid for time and dashed for probability, thicker
line for the metric that run optimized, complete-graph baselines as the residual origin.

Two things are new, because measurement provides them and prediction did not: the shaded
band is the spread across the 20 surviving elites, and the error bars are the standard
error of the measurement. If the trajectory does not clear its own error bars, the run
found nothing.

Watching the *un-optimized* metric is the point of the twin axes: it shows whether driving
fixation time also moved fixation probability.

In [ ]:
for run in RUNS:
    ga_plots.plot_ga_history(run)

## 4. All four runs together

The ML notebook could not draw this: its eight runs optimized eight different model
outputs and were not commensurable. These four optimize two measured quantities in two
directions, so they share axes, and the envelope they trace is the reachable range for
(31, 34) graphs.

In [ ]:
ga_plots.plot_ga_runs_comparison(RUNS);

## 5. Where the winners sit

The figure the experiment exists to produce. N=31 and E=34 are exactly `avian_r4_l7`, and
`mutate_graph` preserves both counts, so every graph the search ever saw is size-matched to
the avian lung graph. This asks: among all connected graphs with the avian graph's node and
edge count, what are the extremes, and where does the real topology sit among them?

In [ ]:
reference_stats = pd.read_csv(REFERENCE_BATCH / "graph_statistics.csv")

for metric in ("mean_steps", "prob_fixation"):
    ga_plots.plot_ga_winners_in_context(RUNS, reference_stats, metric=metric)

## 6. The winning topologies

Measured values, not predictions.

In [ ]:
winners = ga_io.final_population_stats(RUNS)
winners.groupby("run").agg(
    n=("wl_hash", "size"),
    best_mean_steps=("mean_steps", "max"),
    best_prob_fixation=("prob_fixation", "max"),
    worst_mean_steps=("mean_steps", "min"),
    worst_prob_fixation=("prob_fixation", "min"),
)

In [ ]:
# Draw the top few graphs of each run.
N_TO_DRAW = 3

for run in RUNS:
    population = joblib.load(run / "final_population.pkl")
    print(f"\n=== {run.name} ===")
    for graph in population[:N_TO_DRAW]:
        graph.draw(title=f"{graph.name}  ({run.name})")

## 7. ML-driven versus simulation-driven

The ML-driven winners were measured at 100K repeats in
`2026_07_20-extreme_ocmbined_100K-2`. Comparing them against the simulation-driven winners
answers whether the residual predictors were pointing in the right direction, which
`extreme_graphs.ipynb` on its own could never establish: it had no ground truth to check
against.

In [ ]:
ML_BATCH = Path("simulation_data/2026_07_20-extreme_ocmbined_100K-2")

ml_stats = pd.read_csv(ML_BATCH / "graph_statistics.csv")
ml_stats = ml_stats[ml_stats["r"] == 1.1]

comparison = pd.concat([
    ml_stats.assign(source="ML-driven")[
        ["source", "category", "mean_steps", "prob_fixation"]
    ],
    winners.assign(
        source="simulation-driven",
        category=winners["objective"] + " " + winners["metric"],
    )[["source", "category", "mean_steps", "prob_fixation"]],
])
comparison.groupby(["source", "category"])[["mean_steps", "prob_fixation"]].agg(
    ["mean", "max", "min"]
).round(4)

In [ ]:
# The same numbers as the table above, drawn. Bars are the group mean and grow from the
# complete-graph baseline, so bar direction reads directly as amplifier versus suppressor;
# markers are the individual winners, so a group that never converged cannot hide behind
# its mean. Colors come from CATEGORY_COLOR_DICT; marker shape says how a graph was
# selected (circle = LR, square = XGBOOST, filled diamond = simulation).
#
# The trailing semicolon suppresses the returned Figure. Without it the cell renders twice:
# once when the inline backend flushes open figures at the end of the cell, and again when
# IPython displays the returned object as the cell's value.
ga_plots.plot_ml_vs_simulation(RUNS, ml_stats);

In [ ]:
# The two metrics against each other, one point per winner graph, over the cloud of random
# (31, 34) graphs from section 5. The dotted crosshair is the complete graph, so the four
# quadrants are the amplifier / suppressor combinations. `reference_stats` comes from the
# section 5 cell; drop the argument to plot the winners alone.
ga_plots.plot_ml_vs_simulation_scatter(RUNS, ml_stats, reference_stats=reference_stats, logscale=False);

In [ ]:
# Interactive twin of the scatter above. Hover for the graph name and both measured values;
# click a legend entry to isolate a group, which is the thing the static version cannot do
# with 12 overlapping categories. No semicolon here: a plotly Figure has to be returned to
# render, and unlike matplotlib it is only drawn once.
ga_plots.plot_ml_vs_simulation_scatter_plotly(
    RUNS, ml_stats, reference_stats=reference_stats
)

## 8. Was selection still working?

`n_repeats` exists to make the ranking mean something. This checks whether it did, after
the fact and at no simulation cost: every input is already in `ga_history.csv`.

Selection ranks candidates on a *measured* metric, so what it responds to is the true
value plus measurement noise, and the response is proportional to the correlation between
the two: `rho = 1 / sqrt(1 + (SEM/SD)^2)`. It is 1 when measurement is perfect and 0 when
noise has swamped the real spread between graphs.

The right panel says *why* rho moved, and the two metrics move for opposite reasons:

- `prob_fixation` is a ratio, so its SEM `sqrt(p(1-p)/n)` barely shifts as p does. Its rho
  falls only when the population converges and the between-graph spread collapses.
- `mean_steps` is a scale, so its SEM is proportional to the mean. **A run that succeeds
  at maximizing it inflates its own noise floor in step with its own signal**, and can
  lose rho while still visibly improving.

In [ ]:
ga_plots.plot_selection_efficiency(RUNS);

## 9. Do independent repeats find the same thing?

Needs `replicates > 1` in section 1. With a single replicate this still runs and simply
compares the four runs to each other, which is a different (and already answered)
question.

Three layers, cheapest first:

1. **Same fitness?** Left panel. If replicates do not even land at the same value, nothing
   below matters.
2. **Same kind of graph?** Right panel: the structural fingerprint of the final elites,
   z-scored so six properties on incompatible scales share one axis. Replicates that
   converged on the same topology trace the same line.
3. **Do they look the same?** The cell after that draws them.

Deliberately *not* reported: overlap of `wl_hash`. A run visits ~20k topologies out of an
astronomical space, so overlap between independent runs is zero and always will be. It
would only ever say "the runs disagree completely", which is a fact about the size of the
space and not a result.

In [ ]:
ga_plots.plot_replicate_agreement(RUNS);

In [ ]:
# Layer 3: the best graph from each replicate, side by side. Same objective, independent
# searches -- so anything they share is a property of the objective, not of one lucky run.
for run in RUNS:
    best = joblib.load(run / "final_population.pkl")[0]
    best.draw(title=f"{best.name}  ({run.name.split('-', 1)[-1]})")

# The numbers behind the right panel, if you want them as a table.
elite_props = ga_io.final_elite_properties(RUNS)
elite_props.groupby("run")[ga_plots.REPLICATE_PROPERTIES].mean().round(3)